# Chapter 3 — Triton Vector Construction

Part 1: Foundations: what Triton is, how programs run, and how vectors are built.

**Thread.** One idea runs through the chapter: a Triton value is a register tile whose shape and dtype are fixed at trace time.

- A tile is built from static shapes and explicit dtypes, never from runtime integers.
- `tl.arange` supplies lane indices; `tl.full`, `tl.zeros`, and the `*_like` forms supply constants.
- `tl.broadcast_to` changes the logical shape without moving storage; `tl.cast` changes the dtype without changing the shape.
- `BLOCK: tl.constexpr` and `tl.static_range` keep the shape and loop count on the compile-time side of the boundary.

- **Ladder:** concept note → numerical demo → micro-kernel → applied use.

**Study tip.**
- Write the shape and dtype beside every constructor before running a cell.
- Check the small cases: `tl.arange(0, 8)` is `(8,)` `int32`; `tl.full((2, 4), 3.0, tl.float32)` is `(2, 4)` `float32`.
- Check the like-forms: `tl.zeros_like(x)` copies the shape and dtype of `x`.
- Check the cast: `tl.cast(x, tl.float16)` keeps the shape and halves the element width.
- If your paper agrees with the asserted shape and dtype, the rest is bookkeeping.

**Learning objectives**
- Build index, constant, and like-shaped tiles with `tl.arange`, `tl.full`, `tl.zeros`, `tl.ones`, `tl.full_like`, and `tl.zeros_like`.
- State the shape rule and the dtype rule for every constructor, and predict the asserted `shape` and `dtype` before running the cell.
- Expand a size-1 axis with `tl.broadcast_to` and explain why the host view has a zero stride and does not copy.
- Cast between `int32`, `int64`, `float16`, `bfloat16`, and `float32`, and say where precision or range is lost.
- Separate compile-time values (`BLOCK`, `tl.constexpr`, `tl.static_range`) from runtime scalars (`pid`, `n_elements`).
- Write, test, and fuse kernels that construct their own tiles instead of receiving them from the host.

**Navigation:** [Index](../../00_index.ipynb) · [← Previous](../ch02_program_execution/02_complete.ipynb) · [Next →](../../part2_memory/ch04_pointers/04_complete.ipynb)

**Estimated time:** 55–70 min · **Difficulty:** Beginner–Intermediate

**Prerequisites**
- Chapter 2: program IDs, grids, and the `offs = pid * BLOCK + tl.arange(0, BLOCK)` skeleton.
- NumPy or PyTorch broadcasting.
- No prior GPU experience.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

**Connection.** Chapter 2 used the skeleton `offs = pid * BLOCK + tl.arange(0, BLOCK)`; these diagrams isolate the pieces that skeleton depends on.

### Scalar vs vector

```text
scalar s = 3                 vector v = [0 1 2 3]        (shape (4,))
   |                              |
   +-------- v + s --------------> [3 4 5 6]             (shape (4,))
```

- **Rule.** A scalar is rank 0; it broadcasts over a tile without adding a dimension.
- **Example.** A `(4,)` tile plus a scalar stays `(4,)`.

### Compile-time length

```text
kernel source                 trace time                    launch
-------------                 ----------                    ------
tl.arange(0, BLOCK)  ------>  BLOCK is constexpr  ------>  BLOCK=8 or BLOCK=16
                              shape (BLOCK,) is a type     two compiled kernels
```

- **Rule.** `BLOCK` is not a runtime integer.
- **Failure.** Changing `BLOCK` changes the tensor type and triggers a new specialization.

### Dtype promotion

```text
int32 + float32   -> float32       float16 + float32  -> float32
int32 + int64     -> int64         bfloat16 + float32 -> float32
float16 + float16 -> float16       bool + int32       -> int32
```

- **Rule.** The result takes the wider participating type.
- **Triton.** Cast explicitly when the target is a storage type rather than an arithmetic type.

### broadcast_to: a zero-stride view

```text
col (3,1) stride (1,1)          broadcast_to(col, (3,4)) stride (1,0)
+----+                          +----+----+----+----+
| 10 |                          | 10 | 10 | 10 | 10 |
| 20 |   ------------->         | 20 | 20 | 20 | 20 |
| 30 |                          | 30 | 30 | 30 | 30 |
+----+                          +----+----+----+----+
                                   ^
                                   one storage element reused four times
```

- **Rule.** The logical shape grows; the storage does not.
- **Triton.** Inside a kernel, `tl.broadcast_to` is a register-level reinterpretation with the same idea.
- **Bridge.** Part I names every constructor behind these diagrams.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU)
assert hasattr(tl, "arange") and hasattr(tl, "full") and hasattr(tl, "zeros_like")
assert triton.__version__ and torch.__version__

# Host-side contract used by the opening study tip and the concept diagrams.
assert tuple(torch.arange(8, dtype=torch.int32).shape) == (8,)
assert torch.arange(8, dtype=torch.int32).dtype == torch.int32
assert tuple(torch.full((2, 4), 3.0, dtype=torch.float32).shape) == (2, 4)
assert torch.full((2, 4), 3.0, dtype=torch.float32).dtype == torch.float32
assert (torch.tensor([1], dtype=torch.int32) + torch.tensor([1.0])).dtype == torch.float32
assert (torch.tensor([1], dtype=torch.int32) + torch.tensor([1], dtype=torch.int64)).dtype == torch.int64
assert (torch.tensor([1], dtype=torch.float16) + torch.tensor([1], dtype=torch.float32)).dtype == torch.float32
assert (torch.tensor([1], dtype=torch.bfloat16) + torch.tensor([1], dtype=torch.float32)).dtype == torch.float32
assert (torch.tensor([1], dtype=torch.float16) + torch.tensor([1], dtype=torch.float16)).dtype == torch.float16
assert (torch.tensor([True]) + torch.tensor([1], dtype=torch.int32)).dtype == torch.int32

# Part I — Mathematics and concepts

**Connection.** Chapter 2 fixed the launch skeleton; this part fills in the tiles that skeleton moves.

- **Rule.** A Triton **vector** (also called a tile) is a value with a shape tuple and a dtype that are fixed when the kernel is traced.
  - Construction functions allocate register values.
  - They do not read or write global memory.
- **How to read each section.** Look for six things: the signature, the shape rule, the dtype rule, why/where it is used, one Triton hint, and one failure mode.
- **Version note.** This environment runs Triton 3.7.1, which exports `tl.full` and `tl.zeros_like` but **not** `tl.ones` or `tl.full_like`.
  - The sections keep the standard names and give the portable forms `tl.full(shape, 1.0, dtype)` and `tl.full(x.shape, value, x.dtype)`.
  - The host-side PyTorch names `torch.ones`, `torch.full_like`, and `torch.zeros_like` all exist and are used in Method 01.
- **Bridge.** Start with `tl.arange`, the constructor that produces lane indices.

## 3.1 `tl.arange()`

**Connection.** Builds on Chapter 2's offset skeleton; this is the first constructor that makes a tile.
- **Rule.** `tl.arange(start, end)` returns a 1-D tile of consecutive integers.
- **Formula.** The values are $[s, s+1, \ldots, e-1]$, and the length is $e-s$.
- **Shape.** The result is `(e - s,)`; both bounds are compile-time constants, and the length is a power of two in this Triton version.
- **Dtype.** The result is `int32`; cast to `int64` only when an offset can exceed $2^{31}-1$.
- **Example.** `tl.arange(0, 8)` is `(8,)` with values `(0,1,2,3,4,5,6,7)`.
- **Where.** Every address starts as `base + offset`; `arange` builds offsets and the row and column vectors that feed masks and broadcasts.
- **Triton.** Use `pid * BLOCK + tl.arange(0, BLOCK)`; multiply the tile by a stride only when the stride is a scalar.
- **Failure.** `tl.arange(0, n_elements)` with a runtime `n_elements` cannot trace; round up to `BLOCK`, then mask. A non-power-of-two bound also fails here.
- **Bridge.** `tl.full` reuses the same static-shape idea for a constant.

## 3.2 `tl.full()`

**Connection.** Builds on `tl.arange`: both create a tile with a compile-time shape, but `full` fills it with one value.
- **Rule.** `tl.full(shape, value, dtype)` returns a tile with exactly `shape`.
- **Formula.** Every lane is $c$: $X_{ij}=c$.
- **Shape.** Every dimension is a compile-time power of two in the current backend.
- **Dtype.** The dtype is explicit; the value is converted once, at trace time. A Python `3.0` with `tl.float16` becomes a half-precision constant.
- **Example.** `tl.full((2, 4), 3.0, tl.float32)` is `(2,4)` `float32` with every lane `3`.
- **Where.** Use it for a padding value, a sentinel such as `-inf`, an accumulator seed, or a scale tile; it is also the portable spelling of `tl.ones`.
- **Triton.** `tl.full((BLOCK_M, BLOCK_N), 0.0, tl.float32)` is a zero tile that also fixes the shape of every later expression.
- **Failure.** A shape built from a runtime value, `tl.full((n,), ...)`, cannot be traced; omitting the dtype is a type error, so pass it explicitly.
- **Bridge.** `tl.zeros` specializes `full` to the additive identity.

## 3.3 `tl.zeros()`

**Connection.** Builds on `tl.full`: it is the zero-valued member of the same constant-tile family.
- **Rule.** `tl.zeros(shape, dtype)` returns a tile with exactly `shape`.
- **Formula.** Every lane is zero: $X_{ij}=0$.
- **Shape.** The same power-of-two rule applies to every dimension.
- **Dtype.** The dtype is explicit; a `float32` zero and an `int32` zero are different values.
- **Example.** `tl.zeros((2, 4), tl.float32)` is `(2,4)` `float32` with every lane `0`.
- **Where.** It is the starting value of an accumulator, the empty output of a masked operation, and the additive identity of a reduction.
- **Triton.** `acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)` is the matrix-multiply accumulator pattern.
- **Failure.** A narrow accumulator such as `tl.float16` silently overflows on a long sum; accumulate in `float32` and cast once at the end.
- **Bridge.** `tl.ones` is the multiplicative counterpart of `zeros`.

## 3.4 `tl.ones()`

**Connection.** Builds on `tl.zeros` and `tl.full`: it is the one-valued member of the constant-tile family.
- **Rule.** `tl.ones(shape, dtype)` is the standard name; in Triton 3.7.1 use `tl.full(shape, 1.0, dtype)` as the portable form.
- **Formula.** Every lane is one: $X_{ij}=1$.
- **Shape.** The result has exactly `shape`; every dimension is compile-time.
- **Dtype.** The dtype is explicit; `1.0` in `float16`, `1.0` in `float32`, and `1` in `int32` are three different tiles.
- **Example.** `tl.full((8,), 1.0, tl.float32)` is `(8,)` `float32` with every lane `1`.
- **Where.** Ones are the multiplicative identity for scale tiles, the count in a masked mean, and the value compared against row and column indices to build a causal mask.
- **Triton.** When `tl.ones` is unavailable, `tl.full(shape, 1.0, dtype)` is exactly equivalent and works in every recent version.
- **Failure.** Calling `tl.ones` on an older build raises `AttributeError` at trace time; use the portable form and cast before integer arithmetic.
- **Bridge.** The `*_like` family copies a shape and dtype instead of restating them.

## 3.5 `tl.full_like()`

**Connection.** Builds on `tl.full` and on the idea of an existing tile: it copies shape and dtype from `x`.
- **Rule.** `tl.full_like(x, value)` is the standard name; in Triton 3.7.1 use `tl.full(x.shape, value, x.dtype)`.
- **Formula.** Every lane is $c$, with the type of `x`: $X_{ij}=c$.
- **Shape.** The result has the same shape as `x`; no dimension is added or removed.
- **Dtype.** The result has the same dtype as `x`; `value` is converted to it, so shape and dtype travel together.
- **Example.** For `x` of shape `(8,)` and dtype `float32`, `tl.full(x.shape, -1.0, x.dtype)` is `(8,)` `float32` with every lane `-1`.
- **Where.** It builds a sentinel or padding tile that is type-compatible with a loaded tile, so the next `tl.where` or `maximum` needs no cast.
- **Triton.** `tl.full(x.shape, -float("inf"), x.dtype)` is the padding tile for a masked maximum.
- **Failure.** A value outside the target range rounds to `inf` (or wraps, for integers); with `float16`, `-1e9` is `-inf`, not a large negative number.
- **Bridge.** `tl.zeros_like` is the zero-valued member of the same like-shaped family.

## 3.6 `tl.zeros_like()`

**Connection.** Builds on `tl.full_like`: it copies shape and dtype from `x`, then fills with zero.
- **Rule.** `tl.zeros_like(x)` returns a zero tile with the shape and dtype of `x`.
- **Formula.** Every lane is zero: $X_{ij}=0$.
- **Shape.** The result has the same shape as `x`.
- **Dtype.** The result has the same dtype as `x`; values are not copied from `x`.
- **Example.** For `x` of shape `(8,)` and dtype `float32`, `tl.zeros_like(x)` is `(8,)` `float32` with every lane `0`.
- **Where.** It is the zero of the same type as a tile, used for a masked branch, an additive identity, or an output buffer in registers.
- **Triton.** `zero = tl.zeros_like(x32)` keeps a later `tl.where(keep, y, zero)` type-stable.
- **Failure.** `tl.zeros_like` of a Boolean tile is `False`, not numeric zero. For a reduction's `other`, choose the identity of the reduction: `-inf` for max, `+inf` for min.
- **Bridge.** `tl.broadcast_to` changes the shape of a tile without changing its dtype.

## 3.7 `tl.broadcast_to()`

**Connection.** Builds on `tl.zeros_like` and `tl.full_like`: the shape is taken from an existing tile, but now a size-1 axis is expanded.
- **Rule.** `tl.broadcast_to(x, shape)` aligns `x` and `shape` on the right.
- **Formula.** Each target axis must equal the input axis or the input axis must be `1`; the result has exactly the target shape.
- **Shape.** A column bias `(M,1)` becomes `(M,N)`, a row vector `(1,N)` becomes `(M,N)`, and a scalar offset becomes a full tile.
- **Dtype.** The dtype does not change; broadcasting moves no values and converts nothing.
- **Example.** `(3,1) -> (3,4)` is legal; `(3,) -> (4,)` is not, because the axes are `3` and `4`, neither of which is `1`.
- **Where.** It broadcasts a per-column or per-row bias across a matrix without materializing the repeated values.
- **Triton.** `col = bias[:, None]`, then `tl.broadcast_to(col, (BLOCK_M, BLOCK_N))`; the result is a register view, not a copy.
- **Failure.** On the host, the broadcast view is read-only for in-place writes because several logical lanes share one storage element.
- **Bridge.** `tl.cast` is the other shape-preserving transformation: it changes dtype instead of shape.

## 3.8 `tl.cast()`

**Connection.** Builds on `tl.broadcast_to`: both preserve the tile's logical structure, but `cast` changes the element type.
- **Rule.** `tl.cast(x, dtype)` is the same as `x.to(dtype)`.
- **Formula.** The result is $y_i=\operatorname{cast}_d(x_i)$; the shape is unchanged.
- **Dtype.** The result has the requested dtype. Floating-point casts round; float-to-integer casts truncate toward zero.
- **Example.** Casting `(0, 0.1, 0.2)` from `float32` to `float16` gives `(0, 0.0999755859375, 0.199951171875)`.
- **Where.** Load in `float16` for bandwidth, compute in `float32` for accuracy, store in `float16` for size. Integer offsets stay in `int32` unless the tensor is larger than $2^{31}$ elements.
- **Triton.** `x32 = tl.cast(x, tl.float32)` before arithmetic, then `tl.cast(y, tl.float16)` at the store.
- **Failure.** `float16` overflows above `65504` and has about three decimal digits of precision. A cast is not a bit reinterpretation; use `bitcast=True` only when you deliberately want the same bits read as another type.
- **Bridge.** The next section lays out the dtype menu that `cast` selects from.

## 3.9 Triton dtypes

**Connection.** Builds on `tl.cast`: casting only makes sense against the available dtype set.
- **Rule.** The dtype is part of the tile type and is independent of the shape.
- **Formula.** Mixed integer/float arithmetic promotes to float; two float widths promote to the wider float; two integer widths promote to the wider integer.
- **Example.** The storage widths and limits are:

| dtype | bytes | typical role | key limit |
|---|---:|---|---|
| `tl.int32` | 4 | `tl.arange`, lane indices | offsets up to $2^{31}-1$ |
| `tl.int64` | 8 | very large offsets | twice the register width |
| `tl.float16` | 2 | storage, tensor cores | max `65504`, 10-bit mantissa |
| `tl.bfloat16` | 2 | storage, wide range | 7 explicit mantissa bits, `float32` range |
| `tl.float32` | 4 | accumulation, math | 23 explicit mantissa bits |
| `tl.int1` | 1 bit in registers (1 byte stored) | predicates, masks | not a numeric type |

- **Shape.** A dtype does not change the shape; casting or promotion preserves the tile shape.
- **Where.** The storage dtype is a bandwidth decision; the arithmetic dtype is an accuracy decision. They need not be the same.
- **Triton.** `tl.load` returns the pointer's element type; cast immediately if the arithmetic needs more precision.
- **Failure.** A `float16` multiply can overflow even when the final result would fit; promote before the product, not after.
- **Bridge.** Shape and dtype are both parts of the compile-time tile type, which is the subject of 3.10.

## 3.10 Vector shapes and compile-time dimensions

**Connection.** Builds on 3.2-3.9: every constructor so far took a shape argument, and this section explains what that means to the compiler.
- **Rule.** A tile type is a pair `(shape, dtype)`, and every dimension is a `tl.constexpr` integer.
- **Formula.** The number of logical lanes is $T=\prod_{k=0}^{d-1} B_k$.
- **Shape.** In this backend each dimension is a power of two; the compiler allocates registers and chooses a layout from the static shape.
- **Example.** A `(2, 8)` tile has $2\cdot8=16$ lanes; a `(2, 16)` tile has 32.
- **Where.** `BLOCK`, `BLOCK_M`, and `BLOCK_N` are `tl.constexpr` arguments, not runtime scalars.
- **Triton.** Derive the grid from the runtime length with `triton.cdiv`.
- **Failure.** `tl.zeros((n,), dtype=...)` with a runtime `n` fails to trace; runtime lengths belong in the mask and the grid, never in the tile shape.
- **Bridge.** A scalar is the rank-0 case of the same static type system.

## 3.11 Scalar vs vector operations

**Connection.** Builds on 3.10: a scalar is the rank-0 value that broadcasts against a rank-$d$ tile.
- **Rule.** A scalar is rank 0; a scalar combined with a tile broadcasts over every lane and leaves the tile shape unchanged.
- **Formula.** The elementwise lift is $Y_{ij}=X_{ij}\cdot s$.
- **Dtype.** The scalar is converted to the tile's dtype, or to the promoted type, before the operation.
- **Example.** A `(2,3)` tile times the scalar `2.0` stays `(2,3)`.
- **Where.** `pid`, `n_elements`, `scale`, and `shift` are runtime scalars; `offs` and the loaded tile are vectors. `pid * BLOCK + arange` mixes the two.
- **Triton.** `offs = pid * BLOCK + tl.arange(0, BLOCK)` and `y = x * scale + shift` are the two canonical scalar/vector expressions.
- **Failure.** Do not use a runtime scalar as a shape or as an `arange` bound; scalars are values, shapes are types.
- **Bridge.** A static loop repeats scalar/vector expressions a fixed number of times at trace time.

## 3.12 Static ranges and compile-time constants

**Connection.** Builds on 3.11: the loop body is scalar/vector arithmetic, and `tl.static_range` fixes the trip count at trace time.
- **Rule.** `tl.static_range(start, end, step)` unrolls a fixed loop; `K: tl.constexpr` supplies the bound.
- **Formula.** A three-step Horner loop computes $y_{n+1}=a\,y_n+b$ for $n=0,\ldots,K-1$.
- **Shape.** The loop does not change the shape of the tile it updates; it unrolls $K$ copies of the body.
- **Dtype.** The loop does not change dtypes; each iteration must keep the accumulator type stable.
- **Example.** The polynomial kernel computes $1+x+x^2+x^3$; at $x=2$ the result is $15$.
- **Where.** Short fixed loops, such as polynomial evaluation, a fixed stencil, or a few refinement steps, are unrolled so the compiler can schedule every load.
- **Triton.** `for k in tl.static_range(0, K):` with `K: tl.constexpr`; a runtime `range` with a runtime bound is not unrolled.
- **Failure.** A large `K` inflates compile time and code size; a bound that depends on a runtime value is not a static range.
- **Bridge.** The decision table and chapter map below collect all twelve constructors into one reference.

## Decision table — which constructor to reach for

**Connection.** The twelve sections reduce to one question: does the tile need new values, a copied shape, a new shape, or a new dtype?

- **Rule.** Use `arange` for *where*, `full` / `zeros` for *what*, `*_like` for *same type as*, `broadcast_to` for *shape without copy*, and `cast` only at a storage boundary.

| Need | Constructor | Why |
|---|---|---|
| lane indices `0..B-1` | `tl.arange(0, B)` | the canonical offset and mask generator |
| a constant tile with a new shape | `tl.full(shape, value, dtype)` | shape and dtype are explicit |
| a zero accumulator | `tl.zeros(shape, dtype)` | starts a sum without a load |
| a one tile | `tl.full(shape, 1.0, dtype)` | portable; `tl.ones` is absent in 3.7.1 |
| a constant with the shape/dtype of `x` | `tl.full(x.shape, value, x.dtype)` | type-stable sentinel or padding |
| a zero with the shape/dtype of `x` | `tl.zeros_like(x)` | type-stable zero branch |
| expand a size-1 axis | `tl.broadcast_to(x, shape)` | zero-stride register view |
| change precision or storage width | `tl.cast(x, dtype)` | elementwise conversion, shape preserved |
| a fixed unrolled loop | `tl.static_range(0, K)` | trace-time unrolling, no runtime branch |

- **Bridge.** The chapter map pairs each constructor with the kernel or demo that uses it.

## Chapter map

**Connection.** Every row points back to one section and forward to the kernel or demo where that constructor appears.

| § | Constructor | Formula/behavior | Shape rule | Kernel/demo |
|---|---|---|---|---|
| **3.1** | `tl.arange()` | $[s,s+1,\ldots,e-1]$ | `(e-s,)`, bounds compile-time | offsets + tail mask |
| **3.2** | `tl.full()` | $X_{ij}=c$ | exactly `shape` | constant tile |
| **3.3** | `tl.zeros()` | $X_{ij}=0$ | exactly `shape` | accumulator seed |
| **3.4** | `tl.ones()` | $X_{ij}=1$ | exactly `shape` | `tl.full(..., 1.0, ...)` |
| **3.5** | `tl.full_like()` | $X_{ij}=c$ | same as `x` | sentinel like a loaded tile |
| **3.6** | `tl.zeros_like()` | $X_{ij}=0$ | same as `x` | zero branch |
| **3.7** | `tl.broadcast_to()` | $Y_{ij}=b_j$ | target shape, size-1 expands | column-bias add |
| **3.8** | `tl.cast()` | $y=\operatorname{cast}_{d}(x)$ | unchanged | masked affine epilogue |
| **3.9** | Triton dtypes | widths and promotion | unchanged | dtype table |
| **3.10** | compile-time dimensions | $T=\prod_k B_k$ | all axes `tl.constexpr` | `BLOCK` grid demo |
| **3.11** | scalar vs vector | $Y_{ij}=X_{ij}s$ | unchanged | `pid * BLOCK + arange` |
| **3.12** | static ranges | unrolled $K$ steps | unchanged | polynomial kernel |
| **★** | causal mask | $[j\le i]$ | `(T,T)` | attention mask |

- **Bridge.** The worked example now fixes one concrete value, shape, and dtype for every constructor.

## Worked numerical example

**Connection.** This is the numerical demo for 3.1-3.8; every value below is asserted by the next code cell.

- **Rule.** Set `BLOCK = 8` and write each result by hand before running the cell.

**Host constructions.**

| expression | shape | dtype | value |
|---|---|---|---|
| `tl.arange(0, 8)` | `(8,)` | `int32` | `(0,1,2,3,4,5,6,7)` |
| `tl.full((8,), 10.0, tl.float32)` | `(8,)` | `float32` | `(10,\ldots,10)` |
| `tl.zeros((8,), tl.float32)` | `(8,)` | `float32` | `(0,\ldots,0)` |
| `tl.full((8,), 1.0, tl.float32)` | `(8,)` | `float32` | `(1,\ldots,1)` |
| `arange + full + zeros + ones` | `(8,)` | `float32` | `(11,\ldots,18)` |
| `tl.full((2, 4), 3.0, tl.float32)` | `(2,4)` | `float32` | all `3` |
| `tl.zeros((2, 4), tl.float32)` | `(2,4)` | `float32` | all `0` |
| `tl.full(x.shape, -1.0, x.dtype)` | `(8,)` | `float32` | all `-1` |
| `tl.zeros_like(x)` | `(8,)` | `float32` | all `0` |
| `tl.broadcast_to(col, (4,4))` | `(4,4)` | `float32` | rows `5,6,7,8` |
| `tl.cast(y, tl.float16)` | `(8,)` | `float16` | rounded |
| cast back to `float32` | `(8,)` | `float32` | rounded values |

- **Formula.** The vector construction is $v=(0,\ldots,7)+(10,\ldots,10)+(0,\ldots,0)+(1,\ldots,1)=(11,\ldots,18)$.

**Example — constant matrices.**

| matrix | shape | rows |
|---|---|---|
| $F$ | `(2,4)` | `(3,3,3,3)` and `(3,3,3,3)` |
| $Z$ | `(2,4)` | `(0,0,0,0)` and `(0,0,0,0)` |

**Host broadcast.**
- Column $C=(10,20,30)^T$ has shape $(3,1)$.
- $B=\operatorname{broadcast\_to}(C,(3,4))$ has rows `(10,10,10,10)`, `(20,20,20,20)`, and `(30,30,30,30)`.
- `stride(C) = (1,1)` and `stride(B) = (1,0)`.
- `B.data_ptr() == C.data_ptr()`, so the four columns reuse one storage element and no copy is made.

**Device broadcast.**
- Triton tile axes are powers of two here, so the kernel uses the smallest analogue, $(4,1)\to(4,4)$.
- $C_d=(5,6,7,8)^T$; $B_d$ has rows `(5,5,5,5)`, `(6,6,6,6)`, `(7,7,7,7)`, and `(8,8,8,8)`.

**Cast round trip.**
- $y=0.1\,(0,1,\ldots,7)$ is `float32`.
- $h=\operatorname{fp16}(y)=(0, 0.0999755859375, 0.199951171875, 0.300048828125, 0.39990234375, 0.5, 0.60009765625, 0.7001953125)$.
- Casting $h$ back to `float32` reproduces those eight values exactly; it cannot recover the original `float32` values of $0.1k$.

**Device outputs.**

| tile | shape | dtype | value |
|---|---|---|---|
| `vec` | `(8,)` | `float32` | `(11,\ldots,18)` |
| `full` | `(2,4)` | `float32` | all `3` |
| `zeros` | `(2,4)` | `float32` | all `0` |
| `like` | `(8,)` | `float32` | all `-1` |
| `bcast` | `(4,4)` | `float32` | rows `5,6,7,8` |
| `cast16` | `(8,)` | `float16` | the eight values of $h$ |
| `castback` | `(8,)` | `float32` | the same eight values |

**Where this leads.** Part II turns these constructions into a host reference and then into kernels; Chapter 4 uses the same offsets as memory addresses.

In [ ]:
BLOCK = 8


def host_construct(BLOCK: int = 8) -> dict:
    """Method 01: the same constructions with ordinary PyTorch tensors."""
    idx = torch.arange(BLOCK, dtype=torch.int32)
    full_vec = torch.full((BLOCK,), 10.0, dtype=torch.float32)
    zeros_vec = torch.zeros((BLOCK,), dtype=torch.float32)
    ones_vec = torch.ones((BLOCK,), dtype=torch.float32)
    vec = idx.to(torch.float32) + full_vec + zeros_vec + ones_vec

    full_tile = torch.full((2, 4), 3.0, dtype=torch.float32)
    zeros_tile = torch.zeros((2, 4), dtype=torch.float32)

    x = idx.to(torch.float32)
    full_like = torch.full_like(x, -1.0)
    zeros_like = torch.zeros_like(x)

    col = (torch.arange(1, 4, dtype=torch.float32) * 10.0).reshape(3, 1)
    bcast = torch.broadcast_to(col, (3, 4))

    y = x * 0.1
    h = y.to(torch.float16)
    return {
        "idx": idx, "vec": vec, "full": full_tile, "zeros": zeros_tile,
        "ones": ones_vec, "full_like": full_like, "zeros_like": zeros_like,
        "col": col, "bcast": bcast, "y": y, "h": h, "h_back": h.to(torch.float32),
    }


@triton.jit
def worked_construct_kernel(vec_out, full_out, zeros_out, like_out,
                            cast16_out, castback_out, bcast_out,
                            BLOCK: tl.constexpr):
    idx = tl.arange(0, BLOCK)
    f = tl.full((BLOCK,), 10.0, tl.float32)
    z = tl.zeros((BLOCK,), tl.float32)
    o = tl.full((BLOCK,), 1.0, tl.float32)        # tl.ones equivalent
    x = tl.cast(idx, tl.float32)
    tl.store(vec_out + idx, x + f + z + o)

    rows = tl.arange(0, 2)
    cols = tl.arange(0, 4)
    tl.store(full_out + rows[:, None] * 4 + cols[None, :],
             tl.full((2, 4), 3.0, tl.float32))
    tl.store(zeros_out + rows[:, None] * 4 + cols[None, :],
             tl.zeros((2, 4), tl.float32))

    fl = tl.full(x.shape, -1.0, x.dtype)          # tl.full_like equivalent
    zl = tl.zeros_like(x)
    tl.store(like_out + idx, fl + zl)

    y = x * 0.1
    h = tl.cast(y, tl.float16)
    tl.store(cast16_out + idx, h)
    tl.store(castback_out + idx, tl.cast(h, tl.float32))

    col = tl.full((4, 1), 5.0, tl.float32) + tl.arange(0, 4)[:, None]
    bc = tl.broadcast_to(col, (4, 4))
    rr = tl.arange(0, 4)
    tl.store(bcast_out + rr[:, None] * 4 + rr[None, :], bc)


def worked(BLOCK: int = 8) -> dict:
    vec = torch.empty(BLOCK, dtype=torch.float32, device=DEVICE)
    full_tile = torch.empty((2, 4), dtype=torch.float32, device=DEVICE)
    zeros_tile = torch.empty((2, 4), dtype=torch.float32, device=DEVICE)
    like = torch.empty(BLOCK, dtype=torch.float32, device=DEVICE)
    cast16 = torch.empty(BLOCK, dtype=torch.float16, device=DEVICE)
    castback = torch.empty(BLOCK, dtype=torch.float32, device=DEVICE)
    bcast = torch.empty((4, 4), dtype=torch.float32, device=DEVICE)
    worked_construct_kernel[(1,)](vec, full_tile, zeros_tile, like, cast16,
                                  castback, bcast, BLOCK=BLOCK)
    return {"vec": vec, "full": full_tile, "zeros": zeros_tile, "like": like,
            "cast16": cast16, "castback": castback, "bcast": bcast}


def test_worked_example() -> None:
    h = host_construct(BLOCK)
    assert tuple(h["idx"].shape) == (8,) and h["idx"].dtype == torch.int32
    assert h["idx"].tolist() == list(range(8))
    assert tuple(h["vec"].shape) == (8,) and h["vec"].dtype == torch.float32
    assert h["vec"].tolist() == [float(v) for v in range(11, 19)]
    assert tuple(h["full"].shape) == (2, 4) and h["full"].dtype == torch.float32
    assert h["full"].tolist() == [[3.0] * 4, [3.0] * 4]
    assert tuple(h["zeros"].shape) == (2, 4) and h["zeros"].dtype == torch.float32
    assert h["zeros"].tolist() == [[0.0] * 4, [0.0] * 4]
    assert tuple(h["ones"].shape) == (8,) and h["ones"].dtype == torch.float32
    assert h["ones"].tolist() == [1.0] * 8
    assert tuple(h["full_like"].shape) == (8,)
    assert h["full_like"].dtype == torch.float32
    assert h["full_like"].tolist() == [-1.0] * 8
    assert tuple(h["zeros_like"].shape) == (8,)
    assert h["zeros_like"].dtype == torch.float32
    assert h["zeros_like"].tolist() == [0.0] * 8
    assert tuple(h["col"].shape) == (3, 1) and h["col"].dtype == torch.float32
    assert h["col"].stride() == (1, 1) and h["col"].tolist() == [[10.0], [20.0], [30.0]]
    assert tuple(h["bcast"].shape) == (3, 4) and h["bcast"].dtype == torch.float32
    assert h["bcast"].stride() == (1, 0)
    assert h["bcast"].tolist() == [[10.0] * 4, [20.0] * 4, [30.0] * 4]
    assert tuple(h["y"].shape) == (8,) and h["y"].dtype == torch.float32
    assert tuple(h["h"].shape) == (8,) and h["h"].dtype == torch.float16
    expected_h = [0.0, 0.0999755859375, 0.199951171875, 0.300048828125,
                  0.39990234375, 0.5, 0.60009765625, 0.7001953125]
    assert h["h"].tolist() == expected_h
    assert tuple(h["h_back"].shape) == (8,) and h["h_back"].dtype == torch.float32
    assert h["h_back"].tolist() == expected_h


def test_worked_kernel() -> None:
    require_gpu("worked constructors")
    out = worked(BLOCK)
    assert tuple(out["vec"].shape) == (8,) and out["vec"].dtype == torch.float32
    torch.testing.assert_close(
        out["vec"], torch.tensor([float(v) for v in range(11, 19)], device=DEVICE))
    assert tuple(out["full"].shape) == (2, 4) and out["full"].dtype == torch.float32
    torch.testing.assert_close(
        out["full"], torch.full((2, 4), 3.0, device=DEVICE))
    assert tuple(out["zeros"].shape) == (2, 4) and out["zeros"].dtype == torch.float32
    torch.testing.assert_close(
        out["zeros"], torch.zeros((2, 4), device=DEVICE))
    assert tuple(out["like"].shape) == (8,) and out["like"].dtype == torch.float32
    torch.testing.assert_close(
        out["like"], torch.full((8,), -1.0, device=DEVICE))
    assert tuple(out["bcast"].shape) == (4, 4) and out["bcast"].dtype == torch.float32
    torch.testing.assert_close(
        out["bcast"],
        torch.tensor([[5.0] * 4, [6.0] * 4, [7.0] * 4, [8.0] * 4], device=DEVICE))
    expected_h = torch.tensor(
        [0.0, 0.0999755859375, 0.199951171875, 0.300048828125,
         0.39990234375, 0.5, 0.60009765625, 0.7001953125],
        dtype=torch.float16, device=DEVICE)
    assert tuple(out["cast16"].shape) == (8,) and out["cast16"].dtype == torch.float16
    assert tuple(out["castback"].shape) == (8,) and out["castback"].dtype == torch.float32
    torch.testing.assert_close(out["cast16"], expected_h, atol=0, rtol=0)
    torch.testing.assert_close(out["castback"], expected_h.float(), atol=0, rtol=0)


test_worked_example()
if HAS_GPU:
    test_worked_kernel()
print("worked example: shapes, dtypes, and values asserted")

# Part II — Method 01: the host reference

**Connection.** Part I built tiles conceptually and numerically; this part builds the same objects with PyTorch as the correctness specification.

- **Rule.** The host reference is slow and is not a kernel, but every shape, dtype, and value the device kernel produces must match it.
- **Broadcast view.** `torch.broadcast_to` (and `Tensor.expand`) return a view with a zero stride: the logical shape is `(3,4)`, but four adjacent logical columns point at one storage element.
- **No copy.** `data_ptr()` is unchanged and `is_contiguous()` is `False`; calling `.contiguous()` materializes the repeated values and changes the pointer.
- **Formula.** The adjoint of a broadcast is a sum over the expanded axis: if $Y_{ij}=b_i$, then $\partial L/\partial b_i=\sum_{j=0}^{N-1}\partial L/\partial Y_{ij}$.
- **Where.** The autograd cell below checks that rule with a `(3,1) -> (3,4)` view.
- **Bridge.** Part III moves the same constructions into device code.

In [ ]:
def host_broadcast_3x4():
    col = (torch.arange(1, 4, dtype=torch.float32) * 10.0).reshape(3, 1)
    return col, torch.broadcast_to(col, (3, 4))


def test_host_reference() -> None:
    h = host_construct(BLOCK)
    assert tuple(h["vec"].shape) == (8,) and h["vec"].dtype == torch.float32
    assert h["vec"].tolist() == [float(v) for v in range(11, 19)]
    assert tuple(h["full"].shape) == (2, 4)
    assert tuple(h["zeros"].shape) == (2, 4)
    assert tuple(h["ones"].shape) == (8,)
    assert tuple(h["full_like"].shape) == (8,)
    assert tuple(h["zeros_like"].shape) == (8,)
    assert h["full_like"].dtype == h["zeros_like"].dtype == torch.float32
    assert tuple(h["h"].shape) == (8,) and h["h"].dtype == torch.float16
    assert tuple(h["col"].shape) == (3, 1) and h["col"].stride() == (1, 1)
    assert tuple(h["bcast"].shape) == (3, 4) and h["bcast"].stride() == (1, 0)


test_host_reference()

In [ ]:
col, view = host_broadcast_3x4()
materialized = view.contiguous()


def test_broadcast_view() -> None:
    assert tuple(col.shape) == (3, 1) and col.dtype == torch.float32
    assert col.stride() == (1, 1) and col.tolist() == [[10.0], [20.0], [30.0]]
    assert tuple(view.shape) == (3, 4) and view.dtype == torch.float32
    assert view.stride() == (1, 0)
    assert view.data_ptr() == col.data_ptr()
    assert not view.is_contiguous()
    assert view.tolist() == [[10.0] * 4, [20.0] * 4, [30.0] * 4]
    assert tuple(materialized.shape) == (3, 4)
    assert materialized.stride() == (4, 1)
    assert materialized.data_ptr() != col.data_ptr()
    assert materialized.is_contiguous()
    assert torch.equal(materialized, view)


def test_autograd_broadcast() -> None:
    c = (torch.arange(1, 4, dtype=torch.float32) * 10.0).reshape(3, 1)
    c.requires_grad_(True)
    b = torch.broadcast_to(c, (3, 4))
    weights = torch.arange(1, 13, dtype=torch.float32).reshape(3, 4)
    (b * weights).sum().backward()
    assert c.grad.tolist() == [[10.0], [26.0], [42.0]]


test_broadcast_view()
test_autograd_broadcast()
print("broadcast_to: zero-stride view, shared storage; contiguous() copies")

# Part III — Method 02: construct tiles inside device code

**Connection.** Part II established the host reference; this part constructs and broadcasts the same tiles inside kernels.

## Two kernels, two jobs
- **Kernel 1.** Constructs its own indices and constants with `tl.arange`, `tl.full`, `tl.zeros`, and the portable ones form, then stores them.
- **Kernel 2.** Uses `tl.broadcast_to` to add a column vector to a matrix during the load/compute/store pass.
- **Skeleton.** Both reuse the Chapter 2 pattern $\text{offs}=\text{pid}\cdot B+\{0,\ldots,B-1\}$, $\text{mask}=\text{offs}\lt N$, and $P=\lceil N/B\rceil$.
- **Tile map.** For $M=5$, $N=7$, `BLOCK_M=4`, and `BLOCK_N=8`, the grid is $2\times1$.
- **Program 0.** Owns rows `0..3` and columns `0..7`.
- **Program 1.** Owns rows `4..7` with row 7 masked; column 7 is masked in both programs.
- **Broadcast.** The column vector is loaded once per program and broadcast across the eight columns in registers.
- **Bridge.** Part IV fuses construction, selection, and casting into one pass.

In [ ]:
@triton.jit
def construct_tile_kernel(out_vec, out_mat, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements

    base = tl.cast(offs, tl.float32)
    vec = (base
           + tl.full((BLOCK,), 100.0, tl.float32)
           + tl.zeros((BLOCK,), tl.float32)
           + tl.full((BLOCK,), 1.0, tl.float32))
    tl.store(out_vec + offs, vec, mask=mask)

    rows = tl.arange(0, 2)
    tile = (tl.full((2, BLOCK), 7.0, tl.float32)
            + tl.zeros((2, BLOCK), tl.float32)
            + base[None, :])
    tl.store(out_mat + rows[:, None] * n_elements + offs[None, :], tile,
             mask=mask[None, :])


def construct_tile(n: int = 13, BLOCK: int = 8):
    out_vec = torch.empty(n, dtype=torch.float32, device=DEVICE)
    out_mat = torch.empty((2, n), dtype=torch.float32, device=DEVICE)
    construct_tile_kernel[(triton.cdiv(n, BLOCK),)](
        out_vec, out_mat, n, BLOCK=BLOCK
    )
    return out_vec, out_mat


def test_construct_kernel() -> None:
    require_gpu("construct tile")
    for BLOCK in (8, 16):
        n = 13
        out_vec, out_mat = construct_tile(n, BLOCK)
        assert tuple(out_vec.shape) == (n,) and out_vec.dtype == torch.float32
        assert tuple(out_mat.shape) == (2, n) and out_mat.dtype == torch.float32
        ref_vec = torch.arange(n, dtype=torch.float32, device=DEVICE) + 101.0
        row = torch.arange(n, dtype=torch.float32, device=DEVICE) + 7.0
        ref_mat = torch.stack([row, row])
        torch.testing.assert_close(out_vec, ref_vec, atol=0, rtol=0)
        torch.testing.assert_close(out_mat, ref_mat, atol=0, rtol=0)


run_if_gpu(test_construct_kernel)

In [ ]:
@triton.jit
def broadcast_add_kernel(a_ptr, col_ptr, out_ptr, M, N,
                         BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)

    a = tl.load(a_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    col = tl.load(col_ptr + rows, mask=rows < M, other=0.0)
    col_tile = tl.broadcast_to(col[:, None], (BLOCK_M, BLOCK_N))
    tl.store(out_ptr + rows[:, None] * N + cols[None, :], a + col_tile, mask=mask)


def broadcast_add(a, col, BLOCK_M: int = 4, BLOCK_N: int = 8):
    a = torch.as_tensor(a, device=DEVICE, dtype=torch.float32).contiguous()
    col = torch.as_tensor(col, device=DEVICE, dtype=torch.float32).contiguous()
    assert a.ndim == 2 and col.shape == (a.shape[0],)
    out = torch.empty_like(a)
    grid = (triton.cdiv(a.shape[0], BLOCK_M), triton.cdiv(a.shape[1], BLOCK_N))
    broadcast_add_kernel[grid](a, col, out, a.shape[0], a.shape[1],
                               BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


def test_broadcast_add() -> None:
    require_gpu("broadcast add")
    a = torch.arange(35, dtype=torch.float32, device=DEVICE).reshape(5, 7)
    col = torch.arange(5, dtype=torch.float32, device=DEVICE) + 10.0
    assert (triton.cdiv(5, 4), triton.cdiv(7, 8)) == (2, 1)
    out = broadcast_add(a, col, BLOCK_M=4, BLOCK_N=8)
    assert tuple(out.shape) == (5, 7) and out.dtype == torch.float32
    torch.testing.assert_close(out, a + col[:, None], atol=0, rtol=0)


run_if_gpu(test_broadcast_add)

# Part IV — Method 03: fusion, casts, and compile-time control

**Connection.** Part III constructed and broadcast tiles; this part fuses those constructions with selection and a dtype cast.

## Masked affine transform

**Connection.** Builds on 3.4, 3.6, and 3.8: a one tile, a zero tile, and a cast in one kernel.
- **Rule.** Construct both a one tile and a zero tile in registers, select with a mask, and cast once at the store.
- **Formula.** $y_i=\operatorname{fp16}(s\,x_i+b)$ when $l\le x_i\le u$, and $y_i=0$ otherwise.
- **Constructors.** `tl.full(x32.shape, 1.0, x32.dtype)` is the type-stable one (the `full_like` role), and `tl.zeros_like(x32)` is the zero branch.
- **Dtype.** The input and output are `float16`; the arithmetic is `float32`, so the kernel reads two bytes, computes in four, and writes two.
- **Where.** This is the standard mixed-precision shape for a masked affine epilogue.
- **Bridge.** The applied epilogue adds a broadcast bias to the same fused pattern.

## Applied epilogue: scale, bias, ReLU, cast

**Connection.** Builds on the masked affine kernel and on 3.7: the bias is broadcast across the matrix tile.
- **Rule.** Add a per-column bias with `tl.broadcast_to`, apply a scalar affine map, clamp at zero, and store `float16`.
- **Formula.** $y_{ij}=\operatorname{fp16}(\max(s\,x_{ij}+b_j,\,0))$.
- **Where.** It is the epilogue of a linear layer or a normalization block, reduced to its construction primitives.
- **Fusion.** One load, one broadcast, one fused expression, one store; the intermediate `float32` tile never reaches memory.
- **Bridge.** The capstone combines the broadcast and the constructor vocabulary into a causal mask.

In [ ]:
@triton.jit
def masked_affine_kernel(x_ptr, out_ptr, n_elements, scale, shift, lo, hi,
                         BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements

    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    x32 = tl.cast(x, tl.float32)
    keep = (x32 >= lo) & (x32 <= hi)
    one = tl.full(x32.shape, 1.0, x32.dtype)      # tl.full_like equivalent
    zero = tl.zeros_like(x32)
    y = tl.where(keep, x32 * scale + shift * one, zero)
    tl.store(out_ptr + offs, tl.cast(y, tl.float16), mask=mask)


def masked_affine(x, scale: float = 2.0, shift: float = 1.0,
                  lo: float = -1.0, hi: float = 1.0, BLOCK: int = 256):
    x = torch.as_tensor(x, device=DEVICE, dtype=torch.float16).contiguous()
    out = torch.empty_like(x)
    if x.numel() == 0:
        return out
    masked_affine_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(), scale, shift, lo, hi, BLOCK=BLOCK
    )
    return out


def test_masked_affine() -> None:
    require_gpu("masked affine")
    x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0, 3.0],
                     dtype=torch.float16, device=DEVICE)
    out = masked_affine(x)
    assert tuple(out.shape) == (6,) and out.dtype == torch.float16
    ref = torch.where((x.float() >= -1.0) & (x.float() <= 1.0),
                      x.float() * 2.0 + 1.0, torch.zeros_like(x.float()))
    torch.testing.assert_close(out, ref.half(), atol=0, rtol=0)

    torch.manual_seed(0)
    xr = (torch.randn(1000, device=DEVICE) * 2.0).half()
    outr = masked_affine(xr)
    assert tuple(outr.shape) == (1000,) and outr.dtype == torch.float16
    ref = torch.where((xr.float() >= -1.0) & (xr.float() <= 1.0),
                      xr.float() * 2.0 + 1.0, torch.zeros_like(xr.float()))
    torch.testing.assert_close(outr, ref.half(), atol=0, rtol=0)


run_if_gpu(test_masked_affine)

In [ ]:
@triton.jit
def epilogue_kernel(x_ptr, bias_ptr, out_ptr, M, N, scale,
                    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)

    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    bias = tl.load(bias_ptr + cols, mask=cols < N, other=0.0)
    bias_tile = tl.broadcast_to(bias[None, :], (BLOCK_M, BLOCK_N))
    y = tl.cast(x, tl.float32) * scale + bias_tile
    y = tl.where(y > 0.0, y, tl.zeros_like(y))
    tl.store(out_ptr + rows[:, None] * N + cols[None, :],
             tl.cast(y, tl.float16), mask=mask)


def epilogue(x, bias, scale: float = 1.5,
             BLOCK_M: int = 4, BLOCK_N: int = 8):
    x = torch.as_tensor(x, device=DEVICE, dtype=torch.float16).contiguous()
    bias = torch.as_tensor(bias, device=DEVICE, dtype=torch.float16).contiguous()
    assert x.ndim == 2 and bias.shape == (x.shape[1],)
    out = torch.empty_like(x)
    grid = (triton.cdiv(x.shape[0], BLOCK_M), triton.cdiv(x.shape[1], BLOCK_N))
    epilogue_kernel[grid](x, bias, out, x.shape[0], x.shape[1], scale,
                          BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


def test_epilogue() -> None:
    require_gpu("affine epilogue")
    x = torch.tensor([[1.0, -1.0, 0.5], [0.0, 2.0, -3.0]],
                     dtype=torch.float16, device=DEVICE)
    bias = torch.tensor([0.5, -0.5, 1.0], dtype=torch.float16, device=DEVICE)
    out = epilogue(x, bias)
    assert tuple(out.shape) == (2, 3) and out.dtype == torch.float16
    ref = torch.clamp(x.float() * 1.5 + bias.float(), min=0.0).half()
    torch.testing.assert_close(out, ref, atol=0, rtol=0)

    torch.manual_seed(0)
    xr = torch.randn(17, 19, device=DEVICE, dtype=torch.float16)
    br = torch.randn(19, device=DEVICE, dtype=torch.float16)
    outr = epilogue(xr, br)
    assert tuple(outr.shape) == (17, 19) and outr.dtype == torch.float16
    ref = torch.clamp(xr.float() * 1.5 + br.float(), min=0.0).half()
    torch.testing.assert_close(outr, ref, atol=1e-3, rtol=1e-3)


run_if_gpu(test_epilogue)

## Capstone — a causal attention mask from constructors

**Connection.** Combines 3.1, 3.7, 3.10, 3.11, and 3.12 with the fused style of Part IV.

- **Rule.** A decoder attention mask is the predicate $j\le i$ broadcast over a `(T,T)` tile.
- **Formula.** $M_{ij}=\mathbf 1[j\le i]$, where $r_i=i$ and $c_j=j$ are two `tl.arange` vectors; the mask compares their broadcasts.
- **Where.** The result is one tile of zeros and ones, built entirely inside the kernel and used before the softmax of a causal attention block.
- **Example.** For $T=5$ the by-hand mask is lower triangular:

| row $i$ | mask row |
|---|---|
| 0 | `1 0 0 0 0` |
| 1 | `1 1 0 0 0` |
| 2 | `1 1 1 0 0` |
| 3 | `1 1 1 1 0` |
| 4 | `1 1 1 1 1` |

- **Bridge.** The next section makes the compile-time dimension of this tile explicit.

In [ ]:
@triton.jit
def causal_mask_kernel(out_ptr, T, BLOCK: tl.constexpr):
    rows = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    cols = tl.arange(0, BLOCK)
    mask = (rows[:, None] < T) & (cols[None, :] < T)
    causal = cols[None, :] <= rows[:, None]
    values = tl.where(causal, 1.0, 0.0)
    tl.store(out_ptr + rows[:, None] * T + cols[None, :], values, mask=mask)


def causal_mask(T: int = 5, BLOCK: int = 8):
    out = torch.zeros((T, T), dtype=torch.float32, device=DEVICE)
    causal_mask_kernel[(triton.cdiv(T, BLOCK),)](out, T, BLOCK=BLOCK)
    return out


def test_causal_mask() -> None:
    require_gpu("causal mask")
    out = causal_mask(5, BLOCK=8)
    assert tuple(out.shape) == (5, 5) and out.dtype == torch.float32
    ref = torch.tril(torch.ones(5, 5, device=DEVICE))
    torch.testing.assert_close(out, ref, atol=0, rtol=0)
    out = causal_mask(13, BLOCK=16)
    assert tuple(out.shape) == (13, 13) and out.dtype == torch.float32
    ref = torch.tril(torch.ones(13, 13, device=DEVICE))
    torch.testing.assert_close(out, ref, atol=0, rtol=0)


run_if_gpu(test_causal_mask)

## Compile-time dimensions and recompilation

**Connection.** Builds on 3.10 and on the `BLOCK` arguments used by every kernel in Parts III and IV.

- **Rule.** `BLOCK` is a `tl.constexpr`, so `construct_tile_kernel[..., BLOCK=8]` and `construct_tile_kernel[..., BLOCK=16]` are two different compiled kernels.
- **Formula.** The grid is $P=\lceil N/B\rceil$.
- **Example.** For $N=13$, `BLOCK=8` gives two programs and `BLOCK=16` gives one.
- **Where.** The value is part of the specialization key, together with the dtypes and other `constexpr` arguments; runtime `n_elements` changes only the mask and the grid.
- **Triton.** Keep the runtime length in `mask = offs < n_elements` and derive the grid with `triton.cdiv`.
- **Failure.** Changing `BLOCK` compiles a new kernel; caches and timings change with it.
- **Bridge.** The next cells make 3.9-3.12 concrete with a dtype table, a scalar check, and a static-range kernel.

In [ ]:
n = 13
grids = {B: triton.cdiv(n, B) for B in (8, 16, 32)}
tiles = {B: (2, B) for B in (8, 16, 32)}
assert grids == {8: 2, 16: 1, 32: 1}
assert tiles[8] == (2, 8) and tiles[16] == (2, 16) and tiles[32] == (2, 32)
assert (2 * 8, 2 * 16, 2 * 32) == (16, 32, 64)
print("BLOCK=8 -> 2 programs; BLOCK=16 and 32 -> 1 program")

## Putting 3.9–3.12 to work

**Connection.** Completes Part IV by exercising the dtype, scalar, and static-range sections in runnable cells.

- **Dtype table.** The next three cells make 3.9-3.12 concrete: byte widths and `float16` / `bfloat16` limits, a scalar-versus-vector shape check, and a `tl.static_range` polynomial kernel.
- **Warning.** `float16` and `bfloat16` are both two bytes but trade precision for range in opposite directions.
- **float16.** 10-bit mantissa, maximum `65504`.
- **bfloat16.** 7 explicit mantissa bits, exponent range of `float32`.
- **Bridge.** The first code cell checks the dtype table without printing it.

In [ ]:
widths = {
    "int32": (torch.int32, 4),
    "int64": (torch.int64, 8),
    "float16": (torch.float16, 2),
    "bfloat16": (torch.bfloat16, 2),
    "float32": (torch.float32, 4),
    "bool": (torch.bool, 1),
}
for name, (dtype, width) in widths.items():
    assert torch.empty((), dtype=dtype).element_size() == width

mantissa_bits = {"float16": 10, "bfloat16": 7, "float32": 23}
assert torch.finfo(torch.float16).max == 65504.0
assert torch.finfo(torch.bfloat16).max > torch.finfo(torch.float32).max / 2
assert torch.finfo(torch.float32).max > torch.finfo(torch.float16).max
assert all(x is not None for x in
           (tl.int32, tl.int64, tl.float16, tl.bfloat16, tl.float32, tl.int1))
print("dtypes: int32=4 B, int64=8 B, fp16=2 B/10 bits, "
      "bf16=2 B/7 bits, fp32=4 B/23 bits")

In [ ]:
scalar = torch.tensor(2.0)
x = torch.arange(6, dtype=torch.float32).reshape(2, 3)
assert tuple(scalar.shape) == () and scalar.dim() == 0
assert tuple(x.shape) == (2, 3) and x.dtype == torch.float32
assert tuple((x * scalar).shape) == (2, 3)
assert tuple((x + scalar).shape) == (2, 3)

pid, B = 1, 4
offs = pid * B + torch.arange(B, dtype=torch.int32)
assert tuple(offs.shape) == (4,) and offs.dtype == torch.int32
assert offs.tolist() == [4, 5, 6, 7]
assert tuple((offs * 2 + 1).shape) == (4,)
print("scalar: rank 0; pid is a scalar; offs is a (4,) int32 tile")

In [ ]:
@triton.jit
def poly_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.full(x.shape, 1.0, x.dtype)
    for _ in tl.static_range(0, 3):
        y = y * x + 1.0
    tl.store(out_ptr + offs, y, mask=mask)


def poly(x, BLOCK: int = 256):
    x = torch.as_tensor(x, device=DEVICE, dtype=torch.float32).contiguous()
    out = torch.empty_like(x)
    BLOCK = max(BLOCK, triton.next_power_of_2(x.numel()))
    poly_kernel[(triton.cdiv(x.numel(), BLOCK),)](
        x, out, x.numel(), BLOCK=BLOCK
    )
    return out


def test_static_range() -> None:
    require_gpu("static range polynomial")
    x = torch.tensor([2.0, -1.0, 0.5], device=DEVICE)
    out = poly(x, BLOCK=4)
    assert tuple(out.shape) == (3,) and out.dtype == torch.float32
    ref = 1.0 + x + x ** 2 + x ** 3
    torch.testing.assert_close(out, ref, atol=0, rtol=0)
    assert out.tolist() == [15.0, 0.0, 1.875]

    torch.manual_seed(0)
    xr = torch.randn(1000, device=DEVICE)
    outr = poly(xr)
    assert tuple(outr.shape) == (1000,) and outr.dtype == torch.float32
    torch.testing.assert_close(outr, 1.0 + xr + xr ** 2 + xr ** 3)


run_if_gpu(test_static_range)

# Pitfalls, checks, and summary

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| shape/dtype mismatch | a `float16` pointer with a `float32` tile fails or stores garbage | cast the tile to the pointer's dtype at the store |
| `tl.arange` with a runtime bound | tracing fails; the bound is not a type | use `BLOCK: tl.constexpr` and `mask = offs < n` |
| non-power-of-two tile dimension | trace error in this backend | pad to the next power of two and mask |
| `broadcast_to` treated as a copy | the host view is read-only for in-place writes; the kernel still reuses registers | write to a fresh tile, or `.contiguous()` when you really need storage |
| `float16` overflow | values above `65504` become `inf`; a long sum loses precision | accumulate and cast in `float32` |
| implicit promotion surprises | `int32 + float32` becomes `float32`; `int64` offsets double register pressure | cast deliberately at the boundary |
| changing `BLOCK` | a new specialization is compiled; caches and timings change | benchmark the `BLOCK` you ship |
| missing tail mask | out-of-bounds load/store on the last program | mask both the load and the store |
| runtime `n` in a tile shape | `tl.zeros((n,), ...)` cannot trace | keep `n` in the mask and the grid |

## Self-check
1. Write the shape and dtype of `tl.arange(0, 16)`, `tl.full((4, 8), 0.0, tl.float32)`, and `tl.zeros_like(x)` for `x` of shape `(4, 8)` and dtype `float16`.
2. Why does `tl.zeros((n,), dtype=tl.float32)` with a runtime `n` fail, while `tl.zeros((BLOCK,), dtype=tl.float32)` with `BLOCK: tl.constexpr` works?
3. What are the strides of `torch.broadcast_to(col, (3, 4))` when `col` has shape `(3, 1)` and stride `(1, 1)`? Why is the result not contiguous?
4. A `float32` value `70000.0` is cast to `float16`. What is the result, and why is that a range problem rather than a precision problem?
5. Which constructor is the portable spelling of `tl.ones` in Triton 3.7.1? Which is the portable spelling of `tl.full_like`?
6. For `x = 2`, evaluate `1 + x + x^2 + x^3` and the same loop with `tl.static_range(0, 3)`. Why do they agree?
7. In `y = tl.where(keep, x32 * scale + shift * one, zero)`, what are the shapes of `keep`, `one`, and `zero`? Why are they type-stable?
8. What changes when `BLOCK` goes from 8 to 16: the mathematical output, the grid, or the compiled kernel?

## Summary
- Every constructor returns a **register tile** with a compile-time shape and dtype; no constructor touches global memory.
- `tl.arange` is the index generator; `tl.full` / `tl.zeros` / `tl.full(...1.0...)` are the constant generators; `*_like` copy the shape and dtype of an existing tile.
- `tl.broadcast_to` expands only size-1 axes and is a view, not a copy; on the host it has a zero stride, and in a kernel it reuses registers.
- `tl.cast` changes dtype elementwise and preserves shape; store in the pointer's dtype and compute in the accumulator's dtype.
- `BLOCK`, `BLOCK_M`, and `BLOCK_N` are `tl.constexpr`; each value recompiles, while runtime lengths belong in the mask and the grid.
- `tl.static_range` unrolls fixed loops at trace time; a runtime loop does not.
- The capstone builds a causal mask from two `arange` vectors, a broadcast comparison, and `tl.where`.

**Where this leads.** Chapter 4 uses these constructors to build pointers: `base + offset` becomes a memory address, and `BLOCK` becomes the tile that a load or store covers.

# Exercises

1. **Length and dtype.** Build `tl.arange(0, 16)` and print its shape and dtype. Cast it to `int64` and `float32` and print both results. Which cast changes the element width?
2. **Constant family.** Build the same `(4, 8)` tile with `tl.full`, `tl.zeros`, and `tl.full(..., 1.0, ...)`. Confirm that all three have the same shape and that only the values differ.
3. **Like-shaped sentinel.** Load a `float16` tile and build its `-inf` padding tile with `tl.full(x.shape, -float("inf"), x.dtype)`. Compare with `torch.full_like` on the host.
4. **Broadcast shapes.** On the host, test `(3,1)->(3,4)`, `(1,4)->(3,4)`, and `(3,)->(4,)`. Which succeed, and what are the resulting strides?
5. **No copy.** Prove that `torch.broadcast_to` does not copy by comparing `data_ptr()` before and after `.contiguous()`. Explain the pointer change.
6. **Overflow.** Cast `65504.0`, `65520.0`, and `70000.0` from `float32` to `float16`. Which become `inf`, and what does that say about storing an accumulator in `float16`?
7. **Scalar or vector?** In `offs = pid * BLOCK + tl.arange(0, BLOCK)`, classify `pid`, `BLOCK`, `offs`, and the literal `0` as compile-time or runtime, scalar or vector.
8. **Block specialization.** Run the construct kernel with `BLOCK=8` and `BLOCK=16` for `n=13`. Print both grids and both outputs. What is identical and what differs?
9. **Static stencil.** Replace the polynomial with the 5-tap recurrence `y = y * x + c_k` using `tl.static_range(0, 5)`. How many multiply-adds are emitted?
10. **Capstone extension.** Build a causal mask for `T=17` with `BLOCK=32`. Verify the last 15 rows and columns are masked correctly. Then build an anti-diagonal mask with `rows + cols == T - 1`.

In [ ]:
print("Running CPU tests ...")
test_worked_example()
test_host_reference()
test_broadcast_view()
test_autograd_broadcast()

print("\nRunning device tests (GPU, or interpreter with HAS_GPU=True) ...")
for fn in (
    test_worked_kernel,
    test_construct_kernel,
    test_broadcast_add,
    test_masked_affine,
    test_epilogue,
    test_causal_mask,
    test_static_range,
):
    run_if_gpu(fn)